# Download and Process MACA Data

This notebook will download MACAv2 (or v3, if access becomes possible!), check for NaNs, and engineer the same features as the PRISM data.

## Step 1: Set up code space

In [ ]:
# import libraries

# File management
import os
import pathlib
from pathlib import Path
import sys # for importing src files

# Downloading
from tqdm.notebook import tqdm # progress bar
import requests # may need for api access

# Data Management
import pandas as pd
import xarray as xr

# Geospatial data
import geopandas as gpd
import rioxarray as rxr

# Plotting
import matplotlib
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas

print("Imports Complete!")

In [ ]:
# Force Jupyter to reload external modules automatically before executing a cell
# I used Gemini to help me set up the connection to my src files

# force reimporting of src scripts
%load_ext autoreload
%autoreload 2

# Find the parent directory of this notebook (the repo root) and add it to Python's search path
repo_root = str(Path.cwd().parent)
if repo_root not in sys.path:
    sys.path.append(repo_root)

# Now Python can see the src folder
# import just specific functions
from src import maca, features

print('src import complete!')

In [ ]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')
os.makedirs(proj_dir, exist_ok=True)

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
os.makedirs(raw_data_dir, exist_ok=True)

cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')
os.makedirs(raw_data_dir, exist_ok=True)

In [ ]:
# Import study boundary
mhw_train_path = Path(cleaned_data_dir, 'boundaries', 'mhw_buff_clip_gdf.parquet')
mhw_gdf = gpd.read_parquet(mhw_train_path)

## Step 2: Set up MACA download functions

In [ ]:
# data dir for MACA data

maca_dir = os.path.join(raw_data_dir, 'maca_data')
os.makedirs(maca_dir, exist_ok = True)

# pattern
maca_pattern = os.path.join(maca_dir, '*.nc')

In [ ]:
# Function to download MACA data
def download_maca_data(
        site_gdf, site_name,
        models, rcp_value,
        variables, 
        start_date, end_date,
        maca_dir):

    '''
    Downloads and processes MACA data for a given site.
    Processing includes clipping bounds and rewriting longitude values
    
    Args:
    =====
    site_gdf (geopandas.GeoDataFrame):
        Site boundary, used for clipping
    site_name (str):
        Name of site
    models (list):
        List of models to programmatically download
    rcp_value (str):
        RCP value for MACA url; can be 'historical' as well
    variables (str or list):
        Variable name or list of variables ('pr', 'tasmax', 'tasmin', etc.)
    start_date (str):
        Desired start date of downloaded dataset
        Format: 'YYYY-MM-DD'
    end_date (str):
        Desired end date of downloaded dataset
        Format: 'YYYY-MM-DD'
    maca_dir (str):
        Path for data to be saved
    '''

    # Initialize results list
    result_paths = []

    # count total steps for progress bar
    total_steps = len(models)

    # make a bounding box
    xmin, ymin, xmax, ymax = site_gdf.total_bounds

    # convert lon to 0-360 to match MACAv2 data standards
    min_lon_360 = xmin + 360
    max_lon_360 = xmax + 360

    # set fixed catalog years for the MACA url
    catalog_years = "1950_2005" if rcp_value == "historical" else "2006_2099"

    # download and process files
    with tqdm(total=total_steps, desc=f"Processing {site_name}") as pbar:
        # loop through models
        for model in models:
            for variable in variables:
                opendap_url = (f'http://thredds.northwestknowledge.net:8080/thredds/dodsC/'
                            f'agg_macav2metdata_{variable}_{model}_r1i1p1_{rcp_value}'
                            f'_{catalog_years}_CONUS_daily.nc')

                # set filename and path for model results
                file_name = f'maca_{variable}_{model}_{site_name}_{rcp_value}_{start_date}_{end_date}_CONUS_daily.nc'
                maca_path = os.path.join(maca_dir, file_name)   

                # Download or open file
                if not os.path.exists(maca_path):

                    # Send progress warning
                    print(f"{variable}, {model}, {start_date}_{end_date}, {rcp_value} has not been downloaded.")

                    # Load metadata for dataset
                    lazy_ds = xr.open_dataset(opendap_url, mask_and_scale = True)

                    # subset dataset on the server before downloading
                    subset_ds = lazy_ds.sel(
                        time = slice(start_date, end_date),
                        lon = slice(min_lon_360, max_lon_360),
                        # select lat regardless of direction it's counted
                        lat = slice(ymin, ymax)) if lazy_ds.lat[0] < lazy_ds.lat[-1] else slice(ymax, ymin)

                    # compute request and download subset to RAM
                    maca_360_ds = subset_ds.compute()

                    # Standardize MACA ds lon back to -180 to 180
                    maca_ds = maca_360_ds.assign_coords(lon=((maca_360_ds.lon + 180) % 360) - 180).sortby('lon')

                    # Set spatial dims, CRS, and NaNs
                    maca_ds.rio.write_crs('EPSG:4326', inplace=True)
                    maca_ds.rio.set_spatial_dims(x_dim = 'lon', y_dim = 'lat', inplace=True)
                    # ensure NaNs are read as NaNs
                    if var in maca_ds.data_vars:
                        maca_ds[var].rio.write_nodata(np.nan, inplace=True)

                    # Save MACA data to raw data dir
                    maca_ds.to_netcdf(maca_path)

                    # Progress notification
                    print(f"{variable}, {model}, {start_date}_{end_date}, {rcp_value} has been downloaded!.")
                else:
                    # Notification
                    print(f"{variable}, {model}, {start_date}_{end_date}, {rcp_value} already exists.")

                # append MACA path to results so the datasets can be opened easily
                result_paths.append(maca_path)

                # update pbar
                pbar.update(1)

    return result_paths

Need to list and explain model selection here.

In [ ]:
# Set desired MACA parameters

models = {}
variables = {}

In [ ]:
# Download MACA data

test = download_maca_data(
    mhw_gdf, 'mhw',
    models, 'rcp45', variables,
    '2044-10-01', '2065-06-01',
    maca_dir
)

## Step 3: Clip MACA data to boundary

In [ ]:
# load MACA data into memory

if os.exists(maca_paths):
    read_netcdf
else:
    print('MACA data has not been downloaded. Return to Step 2 and download data.')

## Step 4: Perform Feature Engineering

In [ ]:
# do feature engineering

In [ ]:
# drop cols

# flatten dem ds

In [ ]:
# convert MACA ds to df

In [ ]:
# reset index

In [ ]:
# rename cols

# reorder cols

In [ ]:
# drop NaNs and check

## Step 5: Save cleaned and engineered MACA data for ML usage